# N-BEATS experiment - Walmart Store Sales Forecasting

Person B track (DL + Prophet). Second notebook in the track - reuses the exact
harness proven out in `model_experiment_dlinear.ipynb` (panel build, store-level
covariates, filtering, wrapper class shape), with every bug found and fixed
during that notebook's debugging baked in from the start:

- target interior gaps are interpolated at build time (not left as `NaN`,
  which silently poisons a global model's shared weights across every series)
- covariates are built once per **Store** (not per `Store, Dept`), since they
  don't vary by department and a per-Dept build breaks for any department
  that stops appearing partway through the test period
- short-history series are filtered out *before* fitting, and target/covariate
  scalers are fit on that exact filtered set so `inverse_transform` doesn't
  silently misalign predictions with the wrong series
- a hard `NaN` assertion gates training, so a bad build fails loudly instead
  of producing a silently-corrupted model
- the submission step reconciles against `sampleSubmission.csv` exactly
  (drops any extra predicted rows, fills any missing ones with a store/dept
  mean fallback) instead of trusting the raw model output shape

**What's different from DLinear, and needs a fresh decision here:** N-BEATS'
covariate support is checked programmatically (section 5) rather than assumed
- in most darts versions `NBEATSModel` only supports **past** covariates, not
  future or static ones. That's a real, documentable difference from DLinear:
  N-BEATS can't see known-future signals like `IsHoliday` or calendar
  proximity to holidays when forecasting, which the guide flags as a genuine
  architectural weakness worth comparing in the README.


##  Init cell (Colab-compatible)

Same as `model_experiment_dlinear.ipynb` - copy the real first cell from
`notebooks/EDA.ipynb` here instead of this reconstructed version if you
haven't already standardized on one copy-pasted cell across notebooks.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    REPO_URL = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"  # TODO: fill in
    REPO_DIR = "/content/walmart-sales-forecasting"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         f"{REPO_DIR}/requirements.txt", "--quiet"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "darts[torch]", "--quiet"], check=True)

    os.chdir(f"{REPO_DIR}/notebooks")

    from google.colab import drive
    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    repo_data_dir = Path(REPO_DIR) / "data" / "raw"
    if drive_data_dir.exists():
        subprocess.run(["cp", "-r", f"{drive_data_dir}/.", str(repo_data_dir)], check=True)
    else:
        raise FileNotFoundError(
            f"Expected Drive data folder not found at {drive_data_dir}. "
            "Create it (or add it as a My Drive shortcut) before running this notebook."
        )

sys.path.append(str(Path.cwd().parent))


Mounted at /content/drive


##  Imports

In [2]:
import pickle
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from sklearn.preprocessing import StandardScaler

from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler

from src.preprocessing import load_raw_data, WalmartPreprocessor
from src.features import add_temporal_features
from src.evaluation import weighted_mae  # ASSUMPTION: signature (y_true, y_pred, is_holiday) -> float

pd.set_option("display.max_columns", 50)


##  MLflow tracking store

Same shared DagsHub server as XGBoost/LightGBM (Person A) and DLinear -
this is the single source of truth for cross-model WMAE comparison, so it
must not silently fall back to a local store if auth fails.


### DagsHub credentials

Resolves `MLFLOW_TRACKING_USERNAME`/`MLFLOW_TRACKING_PASSWORD` without ever
putting a token in the notebook or the repo - locally from a gitignored `.env`
in the repo root, on Colab from **Colab Secrets** (key icon, left sidebar):
each teammate adds their own `DAGSHUB_USERNAME`/`DAGSHUB_TOKEN` once, tied to
their own Google account. This must run **before** the `set_tracking_uri` cell
below, otherwise `mlflow.start_run(...)` gets a 403 (anonymous reads are
allowed on the public repo, but creating a run is a write and needs auth).

In [ ]:
def _load_env_file(path):
    """Minimal .env loader (KEY=VALUE per line) - avoids adding python-dotenv
    as a dependency for a two-line credentials file. Uses setdefault so an
    env var already set in the shell (local dev) is never overridden."""
    path = Path(path)
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    # Colab Secrets (key icon, left sidebar) - each teammate adds their own
    # DAGSHUB_USERNAME/DAGSHUB_TOKEN secret, tied to their own Google account.
    # Never stored in the notebook or repo, unlike a hardcoded token would be.
    from google.colab import userdata

    os.environ.setdefault("MLFLOW_TRACKING_USERNAME", userdata.get("DAGSHUB_USERNAME"))
    os.environ.setdefault("MLFLOW_TRACKING_PASSWORD", userdata.get("DAGSHUB_TOKEN"))
else:
    _load_env_file(Path.cwd().parent / ".env")

In [3]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("NBEATS_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns or sqlite - that would desync from Person A's XGBoost/"
        "LightGBM runs and from your own DLinear notebook."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("NBEATS_Training").name)


MLflow tracking URI: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow
Active experiment: NBEATS_Training


##  Load, merge, clean

In [4]:
# 1. Load the raw data as a dictionary / raw container
raw_data_dict = load_raw_data(data_dir="../data/raw")

# 2. Correctly extract the DataFrames by their keys
train_raw = raw_data_dict["train"]
test_raw = raw_data_dict["test"]

# 3. Initialize the preprocessor and run your transformation pass
preprocessor = WalmartPreprocessor(data_dir="../data/raw")
preprocessor.fit(train_raw)

train_clean = preprocessor.transform(train_raw)
test_clean = preprocessor.transform(test_raw)

train_feat = add_temporal_features(train_clean)
test_feat = add_temporal_features(test_clean)

print(train_feat.shape, test_feat.shape)
train_feat.head()

(421570, 23) (115064, 22)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,week_of_year,month,year,days_to_super_bowl,days_to_labor_day,days_to_thanksgiving,days_to_christmas
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,5,2,2010,7,217,294,329
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,6,2,2010,0,210,287,322
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,7,2,2010,-7,203,280,315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,8,2,2010,-14,196,273,308
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,9,3,2010,-21,189,266,301


##  Covariate coverage decision

Same check as DLinear - determined at runtime against `features.csv`, not
assumed. If you already answered this in the DLinear notebook, the result
should be identical here (same underlying data), but re-verify rather than
hardcoding it, in case `features_raw`/`preprocessor.features_` differ.


In [5]:
test_dates = pd.date_range("2012-11-02", "2013-07-26", freq="W-FRI")

macro_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment",
              "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

features_lookup = preprocessor.features_  # ASSUMPTION: cached raw features frame, has Store/Date/<macro_cols>

coverage = {}
for col in macro_cols:
    non_null_dates = set(features_lookup.loc[features_lookup[col].notna(), "Date"])
    coverage[col] = set(test_dates).issubset(non_null_dates)

coverage_df = pd.Series(coverage, name="covers_full_test_range")
print(coverage_df)

calendar_future_cols = ["IsHoliday", "week_of_year", "month", "year",
                         "days_to_super_bowl", "days_to_labor_day",
                         "days_to_thanksgiving", "days_to_christmas"]

FUTURE_COVARIATE_COLS = calendar_future_cols + [c for c in coverage_df.index if coverage_df[c]]
PAST_COVARIATE_COLS = [c for c in coverage_df.index if not coverage_df[c]]
all_covariate_cols = FUTURE_COVARIATE_COLS + PAST_COVARIATE_COLS

print("Future covariates:", FUTURE_COVARIATE_COLS)
print("Past covariates:", PAST_COVARIATE_COLS)


Temperature      True
Fuel_Price       True
CPI              True
Unemployment     True
MarkDown1        True
MarkDown2       False
MarkDown3        True
MarkDown4        True
MarkDown5        True
Name: covers_full_test_range, dtype: bool
Future covariates: ['IsHoliday', 'week_of_year', 'month', 'year', 'days_to_super_bowl', 'days_to_labor_day', 'days_to_thanksgiving', 'days_to_christmas', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown1', 'MarkDown3', 'MarkDown4', 'MarkDown5']
Past covariates: ['MarkDown2']


##  Model definition and covariate support check

**This is the key divergence point from DLinear.** Check what the installed
darts version's `NBEATSModel` actually accepts before assuming - in most
darts releases N-BEATS is a past-covariates-only model (no future, no
static), unlike DLinear's fuller support. If `supports_future_covariates` is
`False` below, every covariate column (including calendar/holiday ones) gets
folded into `past_covariates` instead, since that's the only channel N-BEATS
can use - meaning the model literally cannot see known-future holiday timing
when forecasting. Record this in the README as an architectural limitation,
not a bug.


In [6]:
import torch
from pytorch_lightning.callbacks import EarlyStopping

FREQ = "W-FRI"
inferred_freq = pd.infer_freq(sorted(train_feat["Date"].unique()))
print("Inferred frequency from data:", inferred_freq)
assert inferred_freq in ("W-FRI", None), f"Unexpected frequency: {inferred_freq}"

INPUT_CHUNK_LENGTH = 52
OUTPUT_CHUNK_LENGTH = 39   # covers the full 39-week test horizon
MIN_REQUIRED_LENGTH = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH  # darts needs this much history in every training series
CV_N_EPOCHS = 10     # ceiling per CV fold - kept small since this trains 3 separate models
FINAL_N_EPOCHS = 30  # ceiling for the single final fit on full history

# Same architecture everywhere (every CV fold and the final model) so what gets
# validated is actually what gets deployed - an earlier version of this notebook
# let the final model silently fall back to darts' default (much bigger)
# architecture instead of the one actually evaluated in CV.
NBEATS_ARCH_KWARGS = dict(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    num_stacks=10,      # default 30 - this is the main time/param cost, shrink it first
    num_blocks=1,
    num_layers=2,        # default 4
    layer_widths=128,    # default 256
    random_state=42,
)


def make_accelerator_kwargs(callbacks=None):
    """pl_trainer_kwargs shared by every fit call below, so GPU selection is consistent."""
    backend = "gpu" if torch.cuda.is_available() else "cpu"
    kwargs = {"accelerator": backend, "callbacks": callbacks or []}
    if backend == "gpu":
        kwargs["devices"] = 1
    return kwargs


# Throwaway, unfit instance just to introspect covariate support - cheap (no .fit() call).
_support_probe = NBEATSModel(n_epochs=1, batch_size=32, **NBEATS_ARCH_KWARGS)
SUPPORTS_PAST = _support_probe.supports_past_covariates
SUPPORTS_FUTURE = _support_probe.supports_future_covariates

print("Supports past covariates:", SUPPORTS_PAST)
print("Supports future covariates:", SUPPORTS_FUTURE)
print("Supports static covariates:", _support_probe.supports_static_covariates)

if SUPPORTS_FUTURE:
    NBEATS_PAST_COLS = PAST_COVARIATE_COLS
    NBEATS_FUTURE_COLS = FUTURE_COVARIATE_COLS
else:
    NBEATS_PAST_COLS = all_covariate_cols
    NBEATS_FUTURE_COLS = []

print("Columns fed as past_covariates:", NBEATS_PAST_COLS)
print("Columns fed as future_covariates:", NBEATS_FUTURE_COLS)

Inferred frequency from data: W-FRI
Supports past covariates: True
Supports future covariates: False
Supports static covariates: False
Columns fed as past_covariates: ['IsHoliday', 'week_of_year', 'month', 'year', 'days_to_super_bowl', 'days_to_labor_day', 'days_to_thanksgiving', 'days_to_christmas', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown1', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'MarkDown2']
Columns fed as future_covariates: []


##  Build the darts panel (gap-interpolated target, store-level covariates)

Same corrected harness as DLinear's final working version - built once,
correctly, instead of iterated on in place. Target gaps get linearly
interpolated (interior missing weeks, not 0-sales weeks); covariates are
built once per Store, spanning train+test, immune to any single department's
short test presence.


In [7]:
def build_darts_series(df, value_col="Weekly_Sales", covariate_cols=None, freq=FREQ):
    """
    Convert a long-format (Store, Dept, Date, ...) frame into darts TimeSeries,
    one per (Store, Dept). Target gaps are linearly interpolated (genuine
    missing weeks, not 0-sales weeks). Covariate gaps are ffill/bfill'd.
    """
    covariate_cols = covariate_cols or []
    series_list, covariate_series_list, keys = [], [], []

    for (store, dept), g in df.groupby(["Store", "Dept"]):
        g = g.sort_values("Date")
        full_range = pd.date_range(g["Date"].min(), g["Date"].max(), freq=freq)
        g = g.set_index("Date").reindex(full_range)
        g.index.name = "Date"

        if value_col in g.columns:
            filled = g[value_col].interpolate(method="linear", limit_direction="both")
            series_list.append(TimeSeries.from_series(filled, freq=freq))

        if covariate_cols:
            cov = g[covariate_cols].ffill().bfill()
            covariate_series_list.append(TimeSeries.from_dataframe(cov, freq=freq))

        keys.append((store, dept))

    return series_list, (covariate_series_list or None), keys


def build_store_level_covariates(df, covariate_cols, global_start, global_end, freq=FREQ):
    """One covariate TimeSeries per Store (covariates don't vary by Dept),
    spanning the full global range so it's always long enough for any
    forecast horizon regardless of any single Dept's data gaps."""
    store_level = df.drop_duplicates(subset=["Store", "Date"])[["Store", "Date"] + covariate_cols]
    store_cov = {}
    for store, g in store_level.groupby("Store"):
        g = g.sort_values("Date").set_index("Date")
        full_range = pd.date_range(global_start, global_end, freq=freq)
        g = g.reindex(full_range)
        g.index.name = "Date"
        g[covariate_cols] = g[covariate_cols].ffill().bfill()
        store_cov[store] = TimeSeries.from_dataframe(g[covariate_cols], freq=freq)
    return store_cov


def split_past_future(cov_list, future_cols, past_cols):
    future_list = [ts[future_cols] for ts in cov_list] if future_cols else None
    past_list = [ts[past_cols] for ts in cov_list] if past_cols else None
    return past_list, future_list


target_series, _, series_keys = build_darts_series(train_feat, covariate_cols=[], freq=FREQ)
print(f"Built {len(target_series)} target series for {len(series_keys)} (Store, Dept) pairs.")

full_feat_df = pd.concat([train_feat, test_feat], ignore_index=True).drop_duplicates(
    subset=["Store", "Dept", "Date"]
)
numeric_cols = [c for c in all_covariate_cols if c not in ["Store", "Dept", "Date"]]
df_scaler = StandardScaler()
train_dates = train_feat["Date"].unique()
df_scaler.fit(full_feat_df.loc[full_feat_df["Date"].isin(train_dates), numeric_cols])
full_feat_scaled_df = full_feat_df.copy()
full_feat_scaled_df[numeric_cols] = df_scaler.transform(full_feat_df[numeric_cols])

GLOBAL_START = full_feat_scaled_df["Date"].min()
GLOBAL_END = full_feat_scaled_df["Date"].max()
print(f"Covariates span {GLOBAL_START} -> {GLOBAL_END}")

store_cov_series = build_store_level_covariates(
    full_feat_scaled_df, all_covariate_cols, GLOBAL_START, GLOBAL_END, freq=FREQ
)


Built 3331 target series for 3331 (Store, Dept) pairs.
Covariates span 2010-02-05 00:00:00 -> 2013-07-26 00:00:00


## Time-based CV: expanding-window splits

Uses the real shared `src/validation.py` splitter - same one Person A's
XGBoost/LightGBM notebooks use (`expanding_window_splits`), so WMAE numbers
are directly comparable across all 7 models. An earlier version of this
notebook tried to import a `time_based_split` name that was never actually
defined in `src/validation.py`, so it silently always fell back to a fake
single-holdout split instead of real multi-fold CV - fixed here to import
the real function directly.

Same 3 folds / 13-week validation windows as the XGBoost notebook.


In [8]:
from src.validation import expanding_window_splits, split_series, describe_split

N_SPLITS = 3
VAL_WEEKS = 13
MIN_TRAIN_WEEKS = 52

splits = expanding_window_splits(
    train_feat["Date"], n_splits=N_SPLITS, val_weeks=VAL_WEEKS, min_train_weeks=MIN_TRAIN_WEEKS
)
assert len(splits) == N_SPLITS, "history too short for the requested number of folds"

for i, split in enumerate(splits):
    print(f"fold {i}:", describe_split(split))

fold 0: {'train_start': 'start_of_history', 'train_end': '2012-01-27 00:00:00', 'val_start': '2012-02-03 00:00:00', 'val_end': '2012-04-27 00:00:00', 'val_weeks': 13, 'n_holidays_in_val': 1}
fold 1: {'train_start': 'start_of_history', 'train_end': '2012-04-27 00:00:00', 'val_start': '2012-05-04 00:00:00', 'val_end': '2012-07-27 00:00:00', 'val_weeks': 13, 'n_holidays_in_val': 0}
fold 2: {'train_start': 'start_of_history', 'train_end': '2012-07-27 00:00:00', 'val_start': '2012-08-03 00:00:00', 'val_end': '2012-10-26 00:00:00', 'val_weeks': 13, 'n_holidays_in_val': 1}


## Log covariate/architecture decisions (`NBEATS_Feature_Selection`)

Lightweight run - just records the covariate split, target-gap-fill
strategy, and architecture chosen above as params. No model fit happens
here (an earlier version of this notebook wastefully fit a full model in
this step and then discarded it) - actual training happens in the CV and
Final stages below.


In [9]:
with mlflow.start_run(run_name="NBEATS_Feature_Selection"):
    mlflow.log_params({
        "future_covariates": ",".join(NBEATS_FUTURE_COLS) if NBEATS_FUTURE_COLS else "none (unsupported by NBEATSModel)",
        "past_covariates": ",".join(NBEATS_PAST_COLS),
        "supports_past_covariates": SUPPORTS_PAST,
        "supports_future_covariates": SUPPORTS_FUTURE,
        "scaling": "StandardScaler (covariates, fit on train dates only) + darts Scaler (target, per-series, fit per fold/final)",
        "target_gap_fill": "linear_interpolation (interior gaps only - genuine 0-sales weeks are untouched)",
        "covariate_gap_fill": "ffill/bfill, built once per Store (covariates are Dept-invariant)",
        "n_series_total": len(target_series),
        "min_required_length": MIN_REQUIRED_LENGTH,
        **NBEATS_ARCH_KWARGS,
    })

print("Logged NBEATS_Feature_Selection.")

MlflowException: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''

## Multi-fold expanding-window CV and log `NBEATS_CV`

Refits a fresh `NBEATSModel` per fold (same architecture as `NBEATS_ARCH_KWARGS`
above) on that fold's train series, forecasts the fold's 13-week validation
window, and scores with the competition's `weighted_mae`. Per-fold and
mean/std metrics are logged in a single `NBEATS_CV` run, mirroring the
XGBoost notebook's `log_cv_run` pattern so both tracks' CV numbers are
directly comparable.


In [ ]:
def flatten_series(series_list, keys, value_name):
    rows = []
    for (store, dept), ts in zip(keys, series_list):
        df = ts.to_dataframe().reset_index()
        df.columns = ["Date", value_name]
        df["Store"] = store
        df["Dept"] = dept
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def build_fold_series(split, min_train_len):
    """Per-series train/val TimeSeries for one CV fold, via the shared split_series()
    adapter. Only keeps series with enough training history (>= min_train_len, required
    for NBEATSModel.fit() to form a training sample) AND full val-window coverage
    (len(va) == VAL_WEEKS) - series that end before this fold's val window, or start
    after this fold's train_end, are dropped for that fold."""
    train_list, val_list, cov_list, keys = [], [], [], []
    for ts, key in zip(target_series, series_keys):
        try:
            tr, va = split_series(ts, split)
        except Exception:
            continue
        if len(tr) >= min_train_len and len(va) == VAL_WEEKS:
            train_list.append(tr)
            val_list.append(va)
            cov_list.append(store_cov_series[key[0]])
            keys.append(key)
    return train_list, val_list, cov_list, keys


fold_results = []
scored_folds = []

with mlflow.start_run(run_name="NBEATS_CV"):
    mlflow.log_params({
        "cv_strategy": "expanding_window",
        "n_splits": N_SPLITS,
        "val_weeks": VAL_WEEKS,
        "min_train_weeks": MIN_TRAIN_WEEKS,
        "cv_n_epochs": CV_N_EPOCHS,
        "future_covariates": ",".join(NBEATS_FUTURE_COLS) if NBEATS_FUTURE_COLS else "none (unsupported)",
        "past_covariates": ",".join(NBEATS_PAST_COLS),
        **NBEATS_ARCH_KWARGS,
    })

    for i, split in enumerate(splits):
        tr_list, va_list, cov_list, fold_keys = build_fold_series(split, MIN_REQUIRED_LENGTH)
        print(f"fold {i}: {len(fold_keys)} / {len(target_series)} series usable")

        if not fold_keys:
            print(f"fold {i}: no series with enough history - skipping")
            continue

        fold_scaler = Scaler()
        tr_scaled = fold_scaler.fit_transform(tr_list)
        va_scaled = fold_scaler.transform(va_list)

        past_cov, future_cov = split_past_future(cov_list, NBEATS_FUTURE_COLS, NBEATS_PAST_COLS)

        n_nan = sum(int(np.isnan(ts.values()).sum()) for ts in tr_scaled)
        assert n_nan == 0, f"fold {i}: NaNs present in training target - stop"

        early_stop_fold = EarlyStopping(monitor="val_loss", patience=5, mode="min")
        fold_model = NBEATSModel(
            n_epochs=CV_N_EPOCHS,
            batch_size=512,
            pl_trainer_kwargs=make_accelerator_kwargs([early_stop_fold]),
            **NBEATS_ARCH_KWARGS,
        )
        fold_model.fit(
            series=tr_scaled,
            val_series=va_scaled,
            past_covariates=past_cov if SUPPORTS_PAST else None,
            val_past_covariates=past_cov if SUPPORTS_PAST else None,
            future_covariates=future_cov if SUPPORTS_FUTURE else None,
            val_future_covariates=future_cov if SUPPORTS_FUTURE else None,
        )

        preds_scaled = fold_model.predict(
            n=VAL_WEEKS,
            series=tr_scaled,
            past_covariates=past_cov if SUPPORTS_PAST else None,
            future_covariates=future_cov if SUPPORTS_FUTURE else None,
        )
        preds = fold_scaler.inverse_transform(preds_scaled)

        pred_df = flatten_series(preds, fold_keys, "Weekly_Sales_Pred")
        actual_df = train_feat[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]]
        scored_fold = pred_df.merge(actual_df, on=["Store", "Dept", "Date"], how="inner")
        scored_fold["fold"] = i

        wmae_fold = weighted_mae(scored_fold["Weekly_Sales"], scored_fold["Weekly_Sales_Pred"], scored_fold["IsHoliday"])
        mae_fold = float((scored_fold["Weekly_Sales"] - scored_fold["Weekly_Sales_Pred"]).abs().mean())

        mlflow.log_metric(f"wmae_fold_{i}", wmae_fold)
        mlflow.log_metric(f"mae_fold_{i}", mae_fold)
        fold_results.append({**describe_split(split), "wmae": wmae_fold, "mae": mae_fold, "n_series": len(fold_keys)})
        scored_folds.append(scored_fold)

        print(f"fold {i}: WMAE={wmae_fold:.2f}  MAE={mae_fold:.2f}")

    wmae_mean = float(np.mean([r["wmae"] for r in fold_results]))
    wmae_std = float(np.std([r["wmae"] for r in fold_results]))
    mae_mean = float(np.mean([r["mae"] for r in fold_results]))

    mlflow.log_metric("wmae_mean", wmae_mean)
    mlflow.log_metric("wmae_std", wmae_std)
    mlflow.log_metric("mae_mean", mae_mean)
    mlflow.log_dict({"fold_results": fold_results}, "cv_results.json")

scored = pd.concat(scored_folds, ignore_index=True)
print(f"\nCV WMAE: {wmae_mean:.2f} (+/- {wmae_std:.2f})")
print(f"CV MAE:  {mae_mean:.2f}")

## Diagnostics

Same dashboard as DLinear - aggregate actual vs predicted, error
distribution, holiday vs non-holiday WMAE split (the one that matters most
given N-BEATS can't see future holidays directly), and per-series WMAE to
spot which Store/Dept pairs the model struggles with. Pools predictions
across all 3 CV folds (`scored` from the cell above).


In [ ]:
scored["error"] = scored["Weekly_Sales_Pred"] - scored["Weekly_Sales"]
scored["abs_error"] = scored["error"].abs()
scored["weight"] = scored["IsHoliday"].map({True: 5, False: 1})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

agg = scored.groupby("Date")[["Weekly_Sales", "Weekly_Sales_Pred"]].sum().sort_index()
axes[0, 0].plot(agg.index, agg["Weekly_Sales"], label="Actual", marker="o")
axes[0, 0].plot(agg.index, agg["Weekly_Sales_Pred"], label="Predicted", marker="o")
axes[0, 0].set_title("Aggregate Weekly Sales: Actual vs Predicted (Pooled CV Folds)")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Total Weekly Sales ($)")
axes[0, 0].legend()
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].hist(scored["error"], bins=80, color="steelblue")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_title("Prediction Error Distribution")
axes[0, 1].set_xlabel("Predicted - Actual")
axes[0, 1].set_ylabel("Count")

by_holiday = scored.groupby("IsHoliday").apply(lambda g: (g["abs_error"] * g["weight"]).sum() / g["weight"].sum())
axes[1, 0].bar(["Non-Holiday", "Holiday"], by_holiday.values, color=["steelblue", "orange"])
axes[1, 0].set_title("WMAE: Holiday vs Non-Holiday Weeks")
axes[1, 0].set_ylabel("Weighted MAE")

def series_wmae(g):
    w = g["IsHoliday"].map({True: 5, False: 1})
    return (g["abs_error"] * w).sum() / w.sum()

per_series = scored.groupby(["Store", "Dept"]).apply(series_wmae).sort_values(ascending=False)
axes[1, 1].hist(per_series, bins=80, color="seagreen")
axes[1, 1].set_title("Per-Series WMAE Distribution")
axes[1, 1].set_xlabel("WMAE per (Store, Dept)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("../submissions/nbeats_diagnostics.png", dpi=120)
plt.show()

print("Top 10 worst-performing series (highest WMAE):")
print(per_series.head(10))

## 11. Required `.predict()` wrapper (`DartsForecastPipeline`)

Same shape as DLinear's corrected wrapper, parameterized so it works for
either covariate configuration (with or without future covariate support).


In [ ]:
class DartsForecastPipeline:
    def __init__(self, darts_model, preprocessor, scaler_target, df_scaler,
                 numeric_cols, future_covariate_cols, past_covariate_cols,
                 all_covariate_cols, train_history_df, freq="W-FRI",
                 output_chunk_length=39, min_required_length=91):
        self.darts_model = darts_model
        self.preprocessor = preprocessor
        self.scaler_target = scaler_target
        self.df_scaler = df_scaler
        self.numeric_cols = numeric_cols
        self.future_covariate_cols = future_covariate_cols
        self.past_covariate_cols = past_covariate_cols
        self.all_covariate_cols = all_covariate_cols
        self.train_history_df = train_history_df
        self.freq = freq
        self.output_chunk_length = output_chunk_length
        self.min_required_length = min_required_length

    def predict(self, raw_test_df):
        test_clean = self.preprocessor.transform(raw_test_df)
        test_feat_local = add_temporal_features(test_clean)

        combined = pd.concat(
            [self.train_history_df, test_feat_local], ignore_index=True, sort=False
        ).drop_duplicates(subset=["Store", "Dept", "Date"])
        combined[self.numeric_cols] = self.df_scaler.transform(combined[self.numeric_cols])

        target_hist, _, keys = build_darts_series(
            self.train_history_df, value_col="Weekly_Sales", covariate_cols=[], freq=self.freq
        )
        target_hist_scaled = self.scaler_target.transform(target_hist)

        global_start = combined["Date"].min()
        global_end = combined["Date"].max()
        store_cov = build_store_level_covariates(
            combined, self.all_covariate_cols, global_start, global_end, freq=self.freq
        )

        filtered_target, filtered_cov, filtered_keys = [], [], []
        for tgt, key in zip(target_hist_scaled, keys):
            if len(tgt) >= self.min_required_length:
                filtered_target.append(tgt)
                filtered_cov.append(store_cov[key[0]])
                filtered_keys.append(key)

        past_cov = [ts[self.past_covariate_cols] for ts in filtered_cov] if self.past_covariate_cols else None
        future_cov = [ts[self.future_covariate_cols] for ts in filtered_cov] if self.future_covariate_cols else None

        preds_scaled = self.darts_model.predict(
            n=self.output_chunk_length,
            series=filtered_target,
            past_covariates=past_cov if self.darts_model.supports_past_covariates else None,
            future_covariates=future_cov if self.darts_model.supports_future_covariates else None,
        )
        preds = self.scaler_target.inverse_transform(preds_scaled)

        rows = []
        for (store, dept), ts in zip(filtered_keys, preds):
            df = ts.to_dataframe().reset_index()
            df.columns = ["Date", "Weekly_Sales"]
            df["Store"] = store
            df["Dept"] = dept
            rows.append(df)
        result = pd.concat(rows, ignore_index=True)

        result["Id"] = (
            result["Store"].astype(str) + "_" +
            result["Dept"].astype(str) + "_" +
            result["Date"].dt.strftime("%Y-%m-%d")
        )
        return result[["Id", "Weekly_Sales"]]


## 12. Final fit on full history and log `NBEATS_Final`

Refits on all available train data with a fresh model instance (never reuse
a model object whose `.fit()` already ran, in case training on a bad build
earlier left it poisoned).


In [ ]:
scaler_target_full = Scaler()
target_full_scaled = scaler_target_full.fit_transform(target_series)

final_target, final_cov, final_keys = [], [], []
for tgt, key in zip(target_full_scaled, series_keys):
    if len(tgt) >= MIN_REQUIRED_LENGTH:
        final_target.append(tgt)
        final_cov.append(store_cov_series[key[0]])
        final_keys.append(key)

print(f"Original series: {len(target_full_scaled)}")
print(f"Filtered series for final fit: {len(final_target)}")
print(f"Dropped (too short for input+output chunk length): {len(target_full_scaled) - len(final_target)}")

past_full, future_full = split_past_future(final_cov, NBEATS_FUTURE_COLS, NBEATS_PAST_COLS)

n_nan_targets = sum(int(np.isnan(ts.values()).sum()) for ts in final_target)
n_nan_covs = sum(int(np.isnan(ts.values()).sum()) for ts in final_cov)
print(f"NaN count in target inputs: {n_nan_targets}")
print(f"NaN count in covariate inputs: {n_nan_covs}")
assert n_nan_targets == 0 and n_nan_covs == 0, "NaNs present in final fit inputs - stop"

# Same NBEATS_ARCH_KWARGS as every CV fold - no early stopping here since the final
# fit uses all available history with no held-out val set to monitor val_loss against.
final_model = NBEATSModel(
    n_epochs=FINAL_N_EPOCHS,
    batch_size=512,
    pl_trainer_kwargs=make_accelerator_kwargs(),
    **NBEATS_ARCH_KWARGS,
)
final_model.fit(
    series=final_target,
    past_covariates=past_full if SUPPORTS_PAST else None,
    future_covariates=future_full if SUPPORTS_FUTURE else None,
)

pipeline = DartsForecastPipeline(
    darts_model=final_model,
    preprocessor=preprocessor,
    scaler_target=scaler_target_full,
    df_scaler=df_scaler,
    numeric_cols=numeric_cols,
    future_covariate_cols=NBEATS_FUTURE_COLS,
    past_covariate_cols=NBEATS_PAST_COLS,
    all_covariate_cols=all_covariate_cols,
    train_history_df=train_feat,
    freq=FREQ,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    min_required_length=MIN_REQUIRED_LENGTH,
)

with mlflow.start_run(run_name="NBEATS_Final") as final_run:
    mlflow.log_params({
        "n_epochs": FINAL_N_EPOCHS,
        "future_covariates": ",".join(NBEATS_FUTURE_COLS) if NBEATS_FUTURE_COLS else "none (unsupported)",
        "past_covariates": ",".join(NBEATS_PAST_COLS),
        "n_series_final": len(final_target),
        "cv_strategy": "expanding_window",
        "n_splits": N_SPLITS,
        "val_weeks": VAL_WEEKS,
        **NBEATS_ARCH_KWARGS,
    })
    mlflow.log_metric("cv_wmae_mean_at_selection", wmae_mean)
    mlflow.log_metric("cv_wmae_std_at_selection", wmae_std)
    mlflow.log_metric("cv_mae_mean_at_selection", mae_mean)

    with tempfile.TemporaryDirectory() as tmp:
        pkl_path = f"{tmp}/nbeats_pipeline.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump(pipeline, f)
        mlflow.log_artifact(pkl_path, artifact_path="pipeline")

    final_run_id = final_run.info.run_id

print("Logged NBEATS_Final wrapper artifact, run_id:", final_run_id)

## 13. Generate submission CSV

Reconciles against `sampleSubmission.csv` exactly - drops any predicted rows
beyond what's required, fills any missing ones (dropped short-history
series) with a store/dept mean fallback, so the row count and ID set match
precisely, not just approximately.


In [ ]:
submission = pipeline.predict(test_raw)

sample = pd.read_csv("../data/raw/sampleSubmission.csv")
required_ids = set(sample["Id"])

extra_ids = set(submission["Id"]) - required_ids
print("Extra rows generated beyond what's needed:", len(extra_ids))

submission_filtered = submission[submission["Id"].isin(required_ids)].copy()
missing_ids = required_ids - set(submission_filtered["Id"])
print("Still missing after filtering:", len(missing_ids))

fallback_lookup = train_feat.groupby(["Store", "Dept"])["Weekly_Sales"].mean().to_dict()
global_fallback = train_feat["Weekly_Sales"].mean()

missing_df = pd.DataFrame({"Id": sorted(missing_ids)})
parts = missing_df["Id"].str.rsplit("_", n=1, expand=True)
store_dept = parts[0].str.split("_", n=1, expand=True)
missing_df["Store"] = store_dept[0].astype(int)
missing_df["Dept"] = store_dept[1].astype(int)
missing_df["Weekly_Sales"] = missing_df.apply(
    lambda r: fallback_lookup.get((r["Store"], r["Dept"]), global_fallback), axis=1
)

final_submission = pd.concat(
    [submission_filtered, missing_df[["Id", "Weekly_Sales"]]], ignore_index=True
).sort_values("Id").reset_index(drop=True)

print("Final rows:", len(final_submission), "| Expected:", len(sample))
assert len(final_submission) == len(sample)
assert set(final_submission["Id"]) == required_ids

final_submission.to_csv("../submissions/nbeats_submission.csv", index=False)
final_submission.head()


In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact("../submissions/nbeats_submission.csv")
    mlflow.log_artifact("../submissions/nbeats_diagnostics.png")

print("Submission + diagnostics logged as artifacts on NBEATS_Final (run_id:", final_run_id, ")")